In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as D
import torch.optim as optim


In [ ]:
import torch
import torch.nn as nn
import torch.distributions as D

class ContinuousActor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )

        self.mean = nn.Linear(128, action_dim)
        self.log_std = nn.Linear(128, action_dim)
        print(f" self mean is: {self.mean}, self log_std is: {self.log_std}")

    def forward(self, state):
        x = self.net(state)

        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, -20, 2)  # stability
        std = torch.exp(log_std)

        dist = D.Normal(mean, std)
        print(f" dist is: {dist}")
        return dist


In [16]:
class Critic(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state):
        return self.net(state)


In [17]:
env = gym.make("Pendulum-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

action_low = torch.tensor(env.action_space.low, dtype=torch.float32)
action_high = torch.tensor(env.action_space.high, dtype=torch.float32)


In [18]:
actor = ContinuousActor(state_dim, action_dim)
critic = Critic(state_dim)

actor_opt = optim.Adam(actor.parameters(), lr=3e-4)
critic_opt = optim.Adam(critic.parameters(), lr=1e-3)

gamma = 0.99


 self mean is: Linear(in_features=128, out_features=1, bias=True), self log_std is: Linear(in_features=128, out_features=1, bias=True)


In [6]:
def select_action(state):
    state = torch.tensor(state, dtype=torch.float32)

    dist = actor(state)
    raw_action = dist.rsample()
    action = torch.tanh(raw_action)

    # scale to env bounds
    action_env = action_low + (action + 1) * 0.5 * (action_high - action_low)

    log_prob = dist.log_prob(raw_action).sum()

    return action_env.detach().numpy(), log_prob, action


In [11]:
for episode in range(10):

    state, _ = env.reset()
    done = False
    ep_reward = 0

    log_probs = []
    values = []
    rewards = []

    while not done:
        # print(state)
        action, log_prob, _ = select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = done or truncated

        value = critic(torch.tensor(state, dtype=torch.float32))

        log_probs.append(log_prob)
        values.append(value)
        rewards.append(reward)

        state = next_state
        ep_reward += reward

    # ---- compute returns ----
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)
    values = torch.cat(values).squeeze()

    advantages = returns - values.detach()

    # ---- actor update ----
    actor_loss = -(torch.stack(log_probs) * advantages).mean()
    actor_opt.zero_grad()
    actor_loss.backward()
    actor_opt.step()

    # ---- critic update ----
    critic_loss = nn.MSELoss()(values, returns)
    critic_opt.zero_grad()
    critic_loss.backward()
    critic_opt.step()

    print(f"Episode {episode}, Reward: {ep_reward:.1f}")


Episode 0, Reward: -1580.2
Episode 1, Reward: -1173.4
Episode 2, Reward: -1352.2
Episode 3, Reward: -1319.0
Episode 4, Reward: -860.7
Episode 5, Reward: -1074.3
Episode 6, Reward: -782.6
Episode 7, Reward: -1082.2
Episode 8, Reward: -918.2
Episode 9, Reward: -991.7


In [19]:
state, _ = env.reset()
print(f"state is: {state}")

for i in range(10):
    action, _, _ = select_action(state)


state is: [-0.9560113  -0.29332986 -0.5283838 ]
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], grad_fn=<ViewBackward0>), scale: tensor([0.9457], grad_fn=<ExpBackward0>))
 dist is: Normal(loc: tensor([0.0896], g

In [21]:
state = torch.tensor([0.2, -1.1, 0.5])
print(state)
print(state.shape)

tensor([ 0.2000, -1.1000,  0.5000])
torch.Size([3])


In [22]:
mean = torch.tensor([0.3, -0.7])
log_std = torch.tensor([-0.5, 0.2])
std = torch.exp(log_std)

print("mean:", mean)
print("std:", std)


mean: tensor([ 0.3000, -0.7000])
std: tensor([0.6065, 1.2214])


In [24]:
dist = D.Normal(mean, std)
print(dist)

Normal(loc: torch.Size([2]), scale: torch.Size([2]))


In [26]:
raw_action = dist.rsample()
print("raw_action:", raw_action)
action = torch.tanh(raw_action)
print("tanh(action):", action)



raw_action: tensor([-0.4606, -2.2226])
tanh(action): tensor([-0.4306, -0.9768])


In [27]:
log_prob = dist.log_prob(raw_action)
print("log_prob per dim:", log_prob)
print("sum log_prob:", log_prob.sum())


log_prob per dim: tensor([-1.2053, -1.8959])
sum log_prob: tensor(-3.1013)


In [28]:
entropy = dist.entropy()
print("entropy per dim:", entropy)
print("total entropy:", entropy.sum())


entropy per dim: tensor([0.9189, 1.6189])
total entropy: tensor(2.5379)
